# 2. Construção da camada Silver

## 2.1 Análise inicial da qualidade dos dados

Antes da realização das transformações da camada Silver, foram efetuadas
verificações sobre as tabelas da camada Bronze, com o objetivo de identificar
possíveis valores ausentes, registros duplicados e inconsistências nos dados.

Essa etapa permite definir os tratamentos necessários antes da disponibilização
dos dados em uma estrutura padronizada para análise.

In [0]:
# Carrega as tabelas da camada Bronze para análise e tratamento
df_capes = spark.table("mvp.bronze.orcamento_capes")
df_mec = spark.table("mvp.bronze.orcamento_mec")
df_mec_unidades = spark.table("mvp.bronze.orcamento_mec_unidades")
df_bolsas = spark.table("mvp.bronze.bolsas_capes")

In [0]:
# Verifica a estrutura das tabelas da camada Bronze
df_capes.printSchema()
df_mec.printSchema()
df_mec_unidades.printSchema()
df_bolsas.printSchema()

root
 |-- AN_ORCAMENTO: string (nullable = true)
 |-- CD_UNIDADE_GESTORA: string (nullable = true)
 |-- DS_UNIDADE_GESTORA: string (nullable = true)
 |-- CD_FUNCAO_ORCAMENTARIA: string (nullable = true)
 |-- DS_FUNCAO_ORCAMENTARIA: string (nullable = true)
 |-- CD_SUBFUNCAO_ORCAMENTARIA: string (nullable = true)
 |-- DS_SUBFUNCAO_ORCAMENTARIA: string (nullable = true)
 |-- CD_PROGRAMA_GOVERNO: string (nullable = true)
 |-- DS_PROGRAMA_GOVERNO: string (nullable = true)
 |-- CD_ACAO: string (nullable = true)
 |-- DS_ACAO: string (nullable = true)
 |-- CD_PLANO_ORCAMENTARIO: string (nullable = true)
 |-- DS_PLANO_ORCAMENTARIO: string (nullable = true)
 |-- DS_PROGRAMA_TRABALHO: string (nullable = true)
 |-- CD_PTRES: string (nullable = true)
 |-- CD_PLANO_INTERNO: string (nullable = true)
 |-- DS_PLANO_INTERNO: string (nullable = true)
 |-- CD_MODALIDADE_APLICACAO: string (nullable = true)
 |-- DS_MODALIDADE_APLICACAO: string (nullable = true)
 |-- CD_CATEGORIA_ECONOMICA: string (nullable

In [0]:
# Verifica a dimensão das tabelas da camada Bronze
tabelas = {
    "Orçamento CAPES": df_capes,
    "Orçamento MEC": df_mec,
    "Unidades MEC": df_mec_unidades,
    "Bolsas CAPES": df_bolsas
}

for nome, df in tabelas.items():
    print(f"{nome}:")
    print(f"  Registros: {df.count()}")
    print(f"  Colunas: {len(df.columns)}")
    print()

Orçamento CAPES:
  Registros: 15739
  Colunas: 39

Orçamento MEC:
  Registros: 29
  Colunas: 10

Unidades MEC:
  Registros: 4060
  Colunas: 10

Bolsas CAPES:
  Registros: 103360
  Colunas: 18



In [0]:
# Calcula a quantidade e o percentual de valores ausentes por coluna

from pyspark.sql.functions import col, sum, when
from pyspark.sql.types import StringType

# Cria uma função para analisar os valores ausentes de cada DataFrame.
def resumo_nulos(df, nome):

    # Conta a quantidade total de registros da tabela.
    total = df.count()

    # Cria uma lista para armazenar os resultados.
    resultados = []

    # Percorre todos os atributos da tabela.
    for campo in df.schema.fields:

        # Para atributos do tipo string, verifica valores nulos ou vazios.
        if isinstance(campo.dataType, StringType):
            condicao = col(campo.name).isNull() | (col(campo.name) == "")

        # Para os demais tipos, verifica apenas valores nulos.
        else:
            condicao = col(campo.name).isNull()

        # Conta a quantidade de valores ausentes no atributo.
        qtd = (
            df.select(
                sum(when(condicao, 1).otherwise(0)).alias("qtd")
            )
            .collect()[0]["qtd"]
        )
        # Armazena o nome da coluna, a quantidade e o percentual de valores ausentes.
        resultados.append(
            (
                campo.name,
                qtd,
                round((qtd / total) * 100, 2)
            )
        )


    # Exibe o nome da tabela analisada.
    print(f"\n--- {nome} ---")

    # Exibe os resultados ordenados pelo maior percentual de valores ausentes.
    spark.createDataFrame(
        resultados,
        ["COLUNA", "QTD_AUSENTES", "PERCENTUAL_AUSENTES"]
    ).orderBy(
        col("PERCENTUAL_AUSENTES").desc()
    ).show(truncate=False)

# Executa a análise para cada tabela da camada Bronze.
resumo_nulos(df_capes, "ORÇAMENTO CAPES")
resumo_nulos(df_mec, "ORÇAMENTO MEC")
resumo_nulos(df_mec_unidades, "UNIDADES MEC")
resumo_nulos(df_bolsas, "BOLSAS CAPES")


--- ORÇAMENTO CAPES ---
+-------------------------+------------+-------------------+
|COLUNA                   |QTD_AUSENTES|PERCENTUAL_AUSENTES|
+-------------------------+------------+-------------------+
|VL_DOTACAO_SUPLEMENTAR   |15580       |98.99              |
|VL_CREDITO_BLOQUEADO     |15362       |97.6               |
|VL_DOTACAO_INICIAL       |10783       |68.51              |
|VL_DOTACAO_DISPONIVEL    |10718       |68.1               |
|VL_DESPESAS_PAGAS        |4152        |26.38              |
|VL_EM_LIQUIDADO          |3876        |24.63              |
|VL_EM_A_LIQUIDAR         |1855        |11.79              |
|VL_DESPESAS_EMPENHADAS   |1855        |11.79              |
|DS_NATUREZA_DESPESA      |1026        |6.52               |
|DS_ELEMENTO_DESPESA      |1026        |6.52               |
|CD_ELEMENTO_DESPESA      |1026        |6.52               |
|CD_NATUREZA_DESPESA      |1026        |6.52               |
|CD_PLANO_ORCAMENTARIO    |0           |0.0                |

In [0]:
# Analisa a distribuição dos valores ausentes da CAPES por ano

from pyspark.sql.functions import col, sum, when, count

# Define os atributos orçamentários que serão analisados.
colunas_valores = [
    "VL_DOTACAO_INICIAL",
    "VL_DOTACAO_SUPLEMENTAR",
    "VL_DOTACAO_DISPONIVEL",
    "VL_CREDITO_BLOQUEADO",
    "VL_DESPESAS_EMPENHADAS",
    "VL_EM_A_LIQUIDAR",
    "VL_EM_LIQUIDADO",
    "VL_DESPESAS_PAGAS"
]

# Agrupa os registros por exercício orçamentário.
df_nulos_ano = (
    df_capes
    .groupBy("AN_ORCAMENTO")

    # Conta o total de registros e os valores ausentes em cada atributo.
    .agg(
        count("*").alias("QTD_REGISTROS"),
        *[
            sum(
                when(
                    col(c).isNull() | (col(c) == ""),
                    1
                ).otherwise(0)
            ).alias(f"NULOS_{c}")
            for c in colunas_valores
        ]
    )
    # Ordena os resultados pelo ano.
    .orderBy("AN_ORCAMENTO")
)

# Exibe a quantidade de valores ausentes por exercício.
display(df_nulos_ano)

AN_ORCAMENTO,QTD_REGISTROS,NULOS_VL_DOTACAO_INICIAL,NULOS_VL_DOTACAO_SUPLEMENTAR,NULOS_VL_DOTACAO_DISPONIVEL,NULOS_VL_CREDITO_BLOQUEADO,NULOS_VL_DESPESAS_EMPENHADAS,NULOS_VL_EM_A_LIQUIDAR,NULOS_VL_EM_LIQUIDADO,NULOS_VL_DESPESAS_PAGAS
2013,2422,1558,2399,1549,2410,284,284,519,519
2014,2211,1522,2184,1519,2178,284,284,504,504
2015,1774,1057,1756,1042,1720,187,187,371,459
2016,1952,1322,1933,1321,1922,203,203,418,451
2017,1956,1488,1951,1488,1900,184,184,448,473
2018,2032,1463,2024,1463,1988,220,220,461,520
2019,1888,1434,1876,1434,1854,192,192,408,440
2020,1504,939,1457,902,1390,301,301,747,786


In [0]:
# Verifica a existência de possíveis registros duplicados nas tabelas da camada Bronze
# Cria um dicionário contendo o nome de cada base e seu respectivo DataFrame.
tabelas = {
    "ORÇAMENTO CAPES": df_capes,
    "ORÇAMENTO MEC": df_mec,
    "UNIDADES MEC": df_mec_unidades,
    "BOLSAS CAPES": df_bolsas
}
# Percorre individualmente cada uma das tabelas definidas anteriormente.
for nome, df in tabelas.items():
    # Conta a quantidade total de registros existentes na tabela.
    total = df.count()
    # Seleciona apenas as colunas provenientes dos dados originais.
    colunas_dados = [
        c for c in df.columns
        if c not in ["ARQUIVO_ORIGEM", "DT_INGESTAO"]
    ]
    # Seleciona somente as colunas originais e utiliza distinct() para identificar quantas combinações de registros são únicas.
    distintos = (
        df
        .select(colunas_dados)
        .distinct()
        .count()
    )
    # Calcula a quantidade de possíveis registros duplicados.
    duplicados = total - distintos
    # Exibe os resultados da análise para cada tabela.
    print(f"--- {nome} ---")
    print(f"Registros totais: {total}")
    print(f"Registros distintos: {distintos}")
    print(f"Possíveis duplicados: {duplicados}")
    print()

--- ORÇAMENTO CAPES ---
Registros totais: 15739
Registros distintos: 15731
Possíveis duplicados: 8

--- ORÇAMENTO MEC ---
Registros totais: 29
Registros distintos: 29
Possíveis duplicados: 0

--- UNIDADES MEC ---
Registros totais: 4060
Registros distintos: 4060
Possíveis duplicados: 0

--- BOLSAS CAPES ---
Registros totais: 103360
Registros distintos: 103360
Possíveis duplicados: 0



In [0]:
# Identifica os possíveis registros duplicados na base de orçamento da CAPES
# Seleciona apenas as colunas originais, desconsiderando os metadados técnicos.
colunas_dados_capes = [
    c for c in df_capes.columns
    if c not in ["ARQUIVO_ORIGEM", "DT_INGESTAO"]
]
# Agrupa os registros pelas colunas originais e conta quantas vezes cada combinação aparece.
df_duplicados_capes = (
    df_capes
    .groupBy(colunas_dados_capes)
    .count()
    # Mantém apenas os registros que aparecem mais de uma vez.
    .filter(col("count") > 1)
    # Ordena pelos registros com maior número de ocorrências.
    .orderBy(col("count").desc())
)
# Exibe os possíveis registros duplicados encontrados.
display(df_duplicados_capes)

AN_ORCAMENTO,CD_UNIDADE_GESTORA,DS_UNIDADE_GESTORA,CD_FUNCAO_ORCAMENTARIA,DS_FUNCAO_ORCAMENTARIA,CD_SUBFUNCAO_ORCAMENTARIA,DS_SUBFUNCAO_ORCAMENTARIA,CD_PROGRAMA_GOVERNO,DS_PROGRAMA_GOVERNO,CD_ACAO,DS_ACAO,CD_PLANO_ORCAMENTARIO,DS_PLANO_ORCAMENTARIO,DS_PROGRAMA_TRABALHO,CD_PTRES,CD_PLANO_INTERNO,DS_PLANO_INTERNO,CD_MODALIDADE_APLICACAO,DS_MODALIDADE_APLICACAO,CD_CATEGORIA_ECONOMICA,DS_CATEGORIA_ECONOMICA,CD_GRUPO_DESPESA,DS_GRUPO_DESPESA,CD_ELEMENTO_DESPESA,DS_ELEMENTO_DESPESA,CD_NATUREZA_DESPESA,DS_NATUREZA_DESPESA,CD_FONTE_RECURSO,DS_FONTE_RECURSO,VL_DOTACAO_INICIAL,VL_DOTACAO_SUPLEMENTAR,VL_DOTACAO_DISPONIVEL,VL_CREDITO_BLOQUEADO,VL_DESPESAS_EMPENHADAS,VL_EM_A_LIQUIDAR,VL_EM_LIQUIDADO,VL_DESPESAS_PAGAS,count
2019,154003,FUND.COORD.DE APERF.DE PESSOAL NIVEL SUPERIOR,12,EDUCACAO,364,ENSINO SUPERIOR,2080,EDUCACAO DE QUALIDADE PARA TODOS,0487,CONCESSAO DE BOLSAS DE ESTUDO NO ENSINO SUPERIOR,0002,CONCESSAO DE BOLSAS DE ESTUDOS NO PAIS,12364208004870001,108452,'-8,SEM INFORMACAO,90,APLICACOES DIRETAS,3,DESPESAS CORRENTES,3,OUTRAS DESPESAS CORRENTES,null,null,null,null,08,FUNDO SOCIAL-PARC.DEST.EDUCACAO PUBL.E SAUDE,0.00,null,0.00,null,null,null,null,null,5
2018,154003,FUND.COORD.DE APERF.DE PESSOAL NIVEL SUPERIOR,12,EDUCACAO,364,ENSINO SUPERIOR,2080,EDUCACAO DE QUALIDADE PARA TODOS,0487,CONCESSAO DE BOLSAS DE ESTUDO NO ENSINO SUPERIOR,0002,CONCESSAO DE BOLSAS DE ESTUDOS NO PAIS,12364208004870001,108452,'-8,SEM INFORMACAO,30,TRANSFER. A ESTADOS E AO DISTRITO FEDERAL,3,DESPESAS CORRENTES,3,OUTRAS DESPESAS CORRENTES,null,null,null,null,08,FUNDO SOCIAL-PARC.DEST.EDUCACAO PUBL.E SAUDE,0.00,null,0.00,null,null,null,null,null,2
2018,154003,FUND.COORD.DE APERF.DE PESSOAL NIVEL SUPERIOR,12,EDUCACAO,364,ENSINO SUPERIOR,2080,EDUCACAO DE QUALIDADE PARA TODOS,0487,CONCESSAO DE BOLSAS DE ESTUDO NO ENSINO SUPERIOR,0002,CONCESSAO DE BOLSAS DE ESTUDOS NO PAIS,12364208004870001,108452,'-8,SEM INFORMACAO,90,APLICACOES DIRETAS,3,DESPESAS CORRENTES,3,OUTRAS DESPESAS CORRENTES,null,null,null,null,08,FUNDO SOCIAL-PARC.DEST.EDUCACAO PUBL.E SAUDE,0.00,null,0.00,null,null,null,null,null,2
2016,154003,FUND.COORD.DE APERF.DE PESSOAL NIVEL SUPERIOR,12,EDUCACAO,364,ENSINO SUPERIOR,2080,EDUCACAO DE QUALIDADE PARA TODOS,0487,CONCESSAO DE BOLSAS DE ESTUDO NO ENSINO SUPERIOR,0002,CONCESSAO DE BOLSAS DE ESTUDOS NO PAIS,12364208004870001,108452,OCCCUO9414N,BF SUPERIOR - PROAP,90,APLICACOES DIRETAS,3,DESPESAS CORRENTES,3,OUTRAS DESPESAS CORRENTES,null,null,null,null,12,RECURSOS DEST.A MANUT.E DES.DO ENSINO,0.00,null,0.00,null,null,null,null,null,2
2019,154003,FUND.COORD.DE APERF.DE PESSOAL NIVEL SUPERIOR,12,EDUCACAO,364,ENSINO SUPERIOR,2080,EDUCACAO DE QUALIDADE PARA TODOS,0487,CONCESSAO DE BOLSAS DE ESTUDO NO ENSINO SUPERIOR,0002,CONCESSAO DE BOLSAS DE ESTUDOS NO PAIS,12364208004870001,108452,OCCCUO9414N,BF SUPERIOR - PROAP,90,APLICACOES DIRETAS,3,DESPESAS CORRENTES,3,OUTRAS DESPESAS CORRENTES,null,null,null,null,08,FUNDO SOCIAL-PARC.DEST.EDUCACAO PUBL.E SAUDE,"2,000,000.00",null,"2,000,000.00",null,null,null,null,null,2


In [0]:
%sql

CREATE OR REPLACE TEMPORARY FUNCTION tratar_valor(valor STRING)
RETURNS DECIMAL(18,2)
RETURN
    TRY_CAST(
        CASE
            WHEN TRIM(valor) LIKE '(%)' THEN
                CONCAT(
                    '-',
                    REPLACE(
                        REPLACE(
                            REPLACE(TRIM(valor), '(', ''),
                            ')',
                            ''
                        ),
                        ',',
                        ''
                    )
                )
            ELSE REPLACE(TRIM(valor), ',', '')
        END
        AS DECIMAL(18,2)
    );

In [0]:
%sql

CREATE OR REPLACE TABLE mvp.silver.orcamento_capes
USING DELTA
AS

WITH capes_sem_duplicados AS (

    SELECT *
    FROM (
        SELECT
            *,

            ROW_NUMBER() OVER (
                PARTITION BY
                    AN_ORCAMENTO,
                    CD_UNIDADE_GESTORA,
                    DS_UNIDADE_GESTORA,
                    CD_FUNCAO_ORCAMENTARIA,
                    DS_FUNCAO_ORCAMENTARIA,
                    CD_SUBFUNCAO_ORCAMENTARIA,
                    DS_SUBFUNCAO_ORCAMENTARIA,
                    CD_PROGRAMA_GOVERNO,
                    DS_PROGRAMA_GOVERNO,
                    CD_ACAO,
                    DS_ACAO,
                    CD_PLANO_ORCAMENTARIO,
                    DS_PLANO_ORCAMENTARIO,
                    DS_PROGRAMA_TRABALHO,
                    CD_PTRES,
                    CD_PLANO_INTERNO,
                    DS_PLANO_INTERNO,
                    CD_MODALIDADE_APLICACAO,
                    DS_MODALIDADE_APLICACAO,
                    CD_CATEGORIA_ECONOMICA,
                    DS_CATEGORIA_ECONOMICA,
                    CD_GRUPO_DESPESA,
                    DS_GRUPO_DESPESA,
                    CD_ELEMENTO_DESPESA,
                    DS_ELEMENTO_DESPESA,
                    CD_NATUREZA_DESPESA,
                    DS_NATUREZA_DESPESA,
                    CD_FONTE_RECURSO,
                    DS_FONTE_RECURSO,
                    VL_DOTACAO_INICIAL,
                    VL_DOTACAO_SUPLEMENTAR,
                    VL_DOTACAO_DISPONIVEL,
                    VL_CREDITO_BLOQUEADO,
                    VL_DESPESAS_EMPENHADAS,
                    VL_EM_A_LIQUIDAR,
                    VL_EM_LIQUIDADO,
                    VL_DESPESAS_PAGAS

                ORDER BY DT_INGESTAO
            ) AS rn

        FROM mvp.bronze.orcamento_capes
    )

    WHERE rn = 1
)

SELECT
    CAST(AN_ORCAMENTO AS INT) AS AN_ORCAMENTO,

    CD_UNIDADE_GESTORA,
    DS_UNIDADE_GESTORA,

    CD_FUNCAO_ORCAMENTARIA,
    DS_FUNCAO_ORCAMENTARIA,

    CD_SUBFUNCAO_ORCAMENTARIA,
    DS_SUBFUNCAO_ORCAMENTARIA,

    CD_PROGRAMA_GOVERNO,
    DS_PROGRAMA_GOVERNO,

    CD_ACAO,
    DS_ACAO,

    CD_PLANO_ORCAMENTARIO,
    DS_PLANO_ORCAMENTARIO,

    DS_PROGRAMA_TRABALHO,

    CD_PTRES,

    CD_PLANO_INTERNO,
    DS_PLANO_INTERNO,

    CD_MODALIDADE_APLICACAO,
    DS_MODALIDADE_APLICACAO,

    CD_CATEGORIA_ECONOMICA,
    DS_CATEGORIA_ECONOMICA,

    CD_GRUPO_DESPESA,
    DS_GRUPO_DESPESA,

    CD_ELEMENTO_DESPESA,
    DS_ELEMENTO_DESPESA,

    CD_NATUREZA_DESPESA,
    DS_NATUREZA_DESPESA,

    CD_FONTE_RECURSO,
    DS_FONTE_RECURSO,

    tratar_valor(VL_DOTACAO_INICIAL)
        AS VL_DOTACAO_INICIAL,

    tratar_valor(VL_DOTACAO_SUPLEMENTAR)
        AS VL_DOTACAO_SUPLEMENTAR,

    tratar_valor(VL_DOTACAO_DISPONIVEL)
        AS VL_DOTACAO_DISPONIVEL,

    tratar_valor(VL_CREDITO_BLOQUEADO)
        AS VL_CREDITO_BLOQUEADO,

    tratar_valor(VL_DESPESAS_EMPENHADAS)
        AS VL_DESPESAS_EMPENHADAS,

    tratar_valor(VL_EM_A_LIQUIDAR)
        AS VL_EM_A_LIQUIDAR,

    tratar_valor(VL_EM_LIQUIDADO)
        AS VL_EM_LIQUIDADO,

    tratar_valor(VL_DESPESAS_PAGAS)
        AS VL_DESPESAS_PAGAS,

    ARQUIVO_ORIGEM,
    DT_INGESTAO

FROM capes_sem_duplicados;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS QTD_REGISTROS
FROM mvp.silver.orcamento_capes;

QTD_REGISTROS
15731


In [0]:
%sql

DESCRIBE TABLE mvp.silver.orcamento_capes;

col_name,data_type,comment
AN_ORCAMENTO,int,null
CD_UNIDADE_GESTORA,string,A code designating the managing unit responsible for overseeing the budgetary allocation and expenditures.
DS_UNIDADE_GESTORA,string,The name of the managing unit that is responsible for the implementation and management of budgetary resources.
CD_FUNCAO_ORCAMENTARIA,string,"An identifier for the budgetary function, which categorizes the purpose of the budget allocation."
DS_FUNCAO_ORCAMENTARIA,string,"A description of the budgetary function, providing insight into the specific purpose for which the budget is allocated."
CD_SUBFUNCAO_ORCAMENTARIA,string,"A code representing the sub-function of the budget, allowing for a more granular classification of expenditures."
DS_SUBFUNCAO_ORCAMENTARIA,string,A description of the budgetary sub-function that details the specific area of focus within the broader function.
CD_PROGRAMA_GOVERNO,string,"A unique code for government programs, used to identify and track the individual programs receiving budget allocations."
DS_PROGRAMA_GOVERNO,string,"The name of the government program associated with the budgetary data, clarifying the context of the allocation."
CD_ACAO,string,"An identifier for specific actions related to budget allocations, helping to categorize expenditures further."


In [0]:
%sql

SELECT
    AN_ORCAMENTO,
    VL_DOTACAO_INICIAL,
    VL_DOTACAO_DISPONIVEL,
    VL_DESPESAS_EMPENHADAS
FROM mvp.silver.orcamento_capes
WHERE VL_DOTACAO_INICIAL IS NOT NULL
LIMIT 20;

AN_ORCAMENTO,VL_DOTACAO_INICIAL,VL_DOTACAO_DISPONIVEL,VL_DESPESAS_EMPENHADAS
2013,40620417.28,41910417.28,null
2013,-12505975.00,6795284.00,null
2013,38625959.33,38625959.33,38555503.07
2013,12505975.00,12505975.00,12464644.33
2013,5570400.42,5570400.42,5563224.78
2013,284509.88,284509.88,284509.88
2013,197922.70,197922.70,197023.09
2013,49488316.11,49488316.11,49472108.58
2013,5670550.28,5670550.28,5655795.74
2013,0.00,0.00,null


In [0]:
%sql

CREATE OR REPLACE TABLE mvp.silver.orcamento_mec
USING DELTA
AS

SELECT
    CAST(ANO AS INT) AS ANO,
    ORGAO_ORCAMENTARIO,
    CAST(REPLACE(PROJETO_LEI, '.', '') AS DECIMAL(18,2)) AS PROJETO_LEI,
    CAST(REPLACE(DOTACAO_INICIAL, '.', '') AS DECIMAL(18,2)) AS DOTACAO_INICIAL,
    CAST(REPLACE(DOTACAO_ATUAL, '.', '') AS DECIMAL(18,2)) AS DOTACAO_ATUAL,
    CAST(REPLACE(EMPENHADO, '.', '') AS DECIMAL(18,2)) AS EMPENHADO,
    CAST(REPLACE(LIQUIDADO, '.', '') AS DECIMAL(18,2)) AS LIQUIDADO,
    CAST(REPLACE(PAGO, '.', '') AS DECIMAL(18,2)) AS PAGO,
    ARQUIVO_ORIGEM,
    DT_INGESTAO
FROM mvp.bronze.orcamento_mec
WHERE ANO <> 'Total';

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    ANO,
    ORGAO_ORCAMENTARIO,
    DOTACAO_INICIAL,
    DOTACAO_ATUAL,
    EMPENHADO,
    LIQUIDADO,
    PAGO
FROM mvp.silver.orcamento_mec
ORDER BY ANO;

ANO,ORGAO_ORCAMENTARIO,DOTACAO_INICIAL,DOTACAO_ATUAL,EMPENHADO,LIQUIDADO,PAGO
2000,26000 - Ministério da Educação,12638403595.00,14359294228.00,13385729242.00,13385729242.00,11541552456.00
2001,26000 - Ministério da Educação,16163900856.00,16760032191.00,14485011306.00,14485011306.00,13234679546.00
2002,26000 - Ministério da Educação,17421387553.00,18261210134.00,16659309712.00,16659309712.00,15760729579.00
2003,26000 - Ministério da Educação,18037343186.00,18902857426.00,18118944472.00,18117050992.00,16570771043.00
2004,26000 - Ministério da Educação,17303144820.00,19521090512.00,18388018034.00,18388018034.00,17157878368.00
2005,26000 - Ministério da Educação,21022574093.00,22329424928.00,20028496794.00,20028496794.00,18082501460.00
2006,26000 - Ministério da Educação,21671079236.00,24904363112.00,23925513623.00,23925513474.00,21673566266.00
2007,26000 - Ministério da Educação,27579847716.00,29474812754.00,28713160322.00,26020635780.00,24625885394.00
2008,26000 - Ministério da Educação,31714041624.00,34820102236.00,33014256077.00,29854938336.00,28984386427.00
2009,26000 - Ministério da Educação,40524634534.00,45283180939.00,41798912405.00,37380465734.00,36388742029.00


In [0]:
%sql

CREATE OR REPLACE TABLE mvp.silver.orcamento_mec_unidades
USING DELTA
AS

SELECT
    CAST(ANO AS INT) AS ANO,
    UNIDADE_ORCAMENTARIA,
    CAST(REPLACE(PROJETO_LEI, '.', '') AS DECIMAL(18,2)) AS PROJETO_LEI,
    CAST(REPLACE(DOTACAO_INICIAL, '.', '') AS DECIMAL(18,2)) AS DOTACAO_INICIAL,
    CAST(REPLACE(DOTACAO_ATUAL, '.', '') AS DECIMAL(18,2)) AS DOTACAO_ATUAL,
    CAST(REPLACE(EMPENHADO, '.', '') AS DECIMAL(18,2)) AS EMPENHADO,
    CAST(REPLACE(LIQUIDADO, '.', '') AS DECIMAL(18,2)) AS LIQUIDADO,
    CAST(REPLACE(PAGO, '.', '') AS DECIMAL(18,2)) AS PAGO,
    ARQUIVO_ORIGEM,
    DT_INGESTAO
FROM mvp.bronze.orcamento_mec_unidades
WHERE ANO <> 'Total';

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    ANO,
    UNIDADE_ORCAMENTARIA,
    DOTACAO_INICIAL,
    DOTACAO_ATUAL,
    EMPENHADO,
    LIQUIDADO,
    PAGO
FROM mvp.silver.orcamento_mec_unidades
WHERE ANO BETWEEN 2013 AND 2020
ORDER BY ANO, UNIDADE_ORCAMENTARIA;

ANO,UNIDADE_ORCAMENTARIA,DOTACAO_INICIAL,DOTACAO_ATUAL,EMPENHADO,LIQUIDADO,PAGO
2013,26101 - Ministério da Educação - Administração Direta,2640187345.00,2131458406.00,1582219966.00,1245855630.00,1219510136.00
2013,26104 - Instituto Nacional de Educação de Surdos,115694837.00,149141585.00,92368573.00,77666478.00,77498555.00
2013,26105 - Instituto Benjamin Constant,60505502.00,72023130.00,61370306.00,56059294.00,54678411.00
2013,26201 - Colégio Pedro II,388114012.00,465677802.00,448070076.00,412719434.00,410810462.00
2013,26230 - Fundação Universidade Federal do Vale do São Francisco,106015719.00,128502686.00,119906838.00,102751695.00,101987871.00
2013,26231 - Universidade Federal de Alagoas,474410301.00,597925260.00,569261680.00,532182970.00,517492665.00
2013,26232 - Universidade Federal da Bahia,963020322.00,1191221931.00,1122106987.00,1073213168.00,1057229376.00
2013,26233 - Universidade Federal do Ceará,1006864852.00,1197352000.00,1128442246.00,1042582194.00,1021170465.00
2013,26234 - Universidade Federal do Espírito Santo,562970481.00,720662572.00,660063281.00,627204053.00,624743167.00
2013,26235 - Universidade Federal de Goiás,760975004.00,898104362.00,882217796.00,829556573.00,818526973.00


In [0]:
%sql

CREATE OR REPLACE TABLE mvp.silver.bolsas_capes
USING DELTA
AS

SELECT
    CAST(AN_REFERENCIA AS INT) AS AN_REFERENCIA,
    NM_MUNICIPIO_IES_CAPES,
    SG_UF_IES_CAPES,
    NM_REGIAO_IES_CAPES,
    CD_IES_CAPES,
    NM_IES_CAPES,
    SG_IES_CAPES,
    NM_NATUREZA_JURIDICA,
    SG_PROGRAMA_FOMENTO,
    CD_PROGRAMA,
    NM_PROGRAMA,
    NM_GRANDE_AREA,
    NM_AREA_AVALIACAO,
    NM_AREA_CONHECIMENTO,
    NM_NIVEL_PADRAO,
    CAST(QT_BOLSAS_CONCEDIDAS AS INT) AS QT_BOLSAS_CONCEDIDAS,
    ARQUIVO_ORIGEM,
    DT_INGESTAO
FROM mvp.bronze.bolsas_capes;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    AN_REFERENCIA,
    NM_REGIAO_IES_CAPES,
    SG_UF_IES_CAPES,
    NM_IES_CAPES,
    SG_PROGRAMA_FOMENTO,
    NM_PROGRAMA,
    NM_NIVEL_PADRAO,
    QT_BOLSAS_CONCEDIDAS
FROM mvp.silver.bolsas_capes
ORDER BY AN_REFERENCIA
LIMIT 20;

AN_REFERENCIA,NM_REGIAO_IES_CAPES,SG_UF_IES_CAPES,NM_IES_CAPES,SG_PROGRAMA_FOMENTO,NM_PROGRAMA,NM_NIVEL_PADRAO,QT_BOLSAS_CONCEDIDAS
2013,SUDESTE,MG,UNIVERSIDADE FEDERAL DE ALFENAS,FAPEMIG,Bioci�ncias Aplicadas � Sa�de,P�S-DOUTORADO,1
2013,SUDESTE,MG,UNIVERSIDADE FEDERAL DE ALFENAS,DS,MULTIC�NTRICO EM CI�NCIAS FISIOL�GICAS,DOUTORADO,6
2013,SUDESTE,MG,UNIVERSIDADE FEDERAL DE ALFENAS,DS,NI,MESTRADO,6
2013,SUDESTE,MG,UNIVERSIDADE FEDERAL DE ALFENAS,FAPEMIG,CI�NCIAS FARMAC�UTICAS,P�S-DOUTORADO,1
2013,SUDESTE,MG,UNIVERSIDADE FEDERAL DE ALFENAS,DS,Ci�ncia e Engenharia Ambiental,MESTRADO,2
2013,SUDESTE,MG,UNIVERSIDADE FEDERAL DE ALFENAS,DS,ESTAT�STICA APLICADA E BIOMETRIA,MESTRADO,2
2013,SUDESTE,MG,UNIVERSIDADE FEDERAL DE ALFENAS,DS,F�SICA - UFLA - UFSJ - UNIFAL,MESTRADO,3
2013,SUDESTE,MG,UNIVERSIDADE FEDERAL DE ALFENAS,DS,NI,DOUTORADO,4
2013,SUDESTE,MG,UNIVERSIDADE FEDERAL DE ALFENAS,DS,CI�NCIAS ODONTOL�GICAS,MESTRADO,3
2013,SUDESTE,MG,UNIVERSIDADE FEDERAL DE ALFENAS,DS,CI�NCIA E ENGENHARIA DE MATERIAIS,MESTRADO,4


In [0]:
%sql

SELECT 'ORCAMENTO_CAPES' AS TABELA, COUNT(*) AS QTD_REGISTROS
FROM mvp.silver.orcamento_capes

UNION ALL

SELECT 'ORCAMENTO_MEC', COUNT(*)
FROM mvp.silver.orcamento_mec

UNION ALL

SELECT 'MEC_UNIDADES', COUNT(*)
FROM mvp.silver.orcamento_mec_unidades

UNION ALL

SELECT 'BOLSAS_CAPES', COUNT(*)
FROM mvp.silver.bolsas_capes;

TABELA,QTD_REGISTROS
ORCAMENTO_CAPES,15731
ORCAMENTO_MEC,28
MEC_UNIDADES,4059
BOLSAS_CAPES,103360


In [0]:
%sql

SHOW TABLES IN mvp.silver;

database,tableName,isTemporary
silver,bolsas_capes,false
silver,orcamento_capes,false
silver,orcamento_mec,false
silver,orcamento_mec_unidades,false
